# Chatbot

This notebook uses the same direct Ollama `/api/chat` approach as `../scripts/chat.py`, but presents it as a Jupyter chat panel. It can run as a local text chatbot and optionally speak replies through `sdk_client.Robot`.


In [ ]:
import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
MODULES_DIR = NOTEBOOK_DIR.parent
ROOT_DIR = MODULES_DIR.parent
for path in (str(MODULES_DIR), str(ROOT_DIR), str(MODULES_DIR / "scripts")):
    if path not in sys.path:
        sys.path.insert(0, path)

IFACE = os.environ.get("G1_IFACE", "eth0")
DOMAIN_ID = int(os.environ.get("G1_DOMAIN_ID", "0"))
print(f"Configured for iface={IFACE!r}, domain_id={DOMAIN_ID}.")


Import notebook UI helpers and the standard-library HTTP client used by the Ollama script.


In [ ]:
import json
import string
import threading
import time
import urllib.error
import urllib.request
from difflib import SequenceMatcher

import ipywidgets as widgets
from IPython.display import display

from sdk_client import Robot

try:
    import rclpy
    from rclpy.node import Node
    from rclpy.executors import SingleThreadedExecutor
    from std_msgs.msg import String
    MIC_IMPORT_ERROR = None
except Exception as exc:
    rclpy = None
    Node = object
    SingleThreadedExecutor = None
    String = None
    MIC_IMPORT_ERROR = exc


DEFAULT_SYSTEM_PROMPT = (
    "You are the voice of a Unitree humanoid robot. Chat naturally with nearby people. "
    "Reply in no more than 25 words. "
    "Do not mention that you are a language model. Do not use markdown or hidden reasoning."
)


def clean_reply(text):
    # TODO: Strip hidden thinking blocks and normalize whitespace before displaying or speaking the model reply.
    raise NotImplementedError("Participant exercise: complete clean_reply.")


FILLER_TEXTS = {"ah", "eh", "er", "hmm", "hm", "mm", "uh", "um", "嗯", "呃", "啊"}


def decode_payload(raw):
    # TODO: Parse microphone JSON messages when possible and fall back to a raw text payload.
    raise NotImplementedError("Participant exercise: complete decode_payload.")


def payload_index(payload):
    # TODO: Extract the optional monotonically increasing audio index as an int.
    raise NotImplementedError("Participant exercise: complete payload_index.")


def similar(left, right):
    # TODO: Use a string similarity score to compare normalized phrases.
    raise NotImplementedError("Participant exercise: complete similar.")


def is_filler(text):
    # TODO: Normalize punctuation/case and check the filler-word set.
    raise NotImplementedError("Participant exercise: complete is_filler.")


Configure Ollama and build the chat state. These defaults mirror `../scripts/chat.py`; override them with environment variables before running the cell.


In [ ]:
def ollama_tags(base_url, timeout=1.5):
    # TODO: Send a GET request to the Ollama /api/tags endpoint and decode the JSON response.
    raise NotImplementedError("Participant exercise: complete ollama_tags.")


def choose_ollama_endpoint():
    # TODO: Try the configured and fallback Ollama URLs until one responds with tags.
    raise NotImplementedError("Participant exercise: complete choose_ollama_endpoint.")


def choose_model(tags):
    # TODO: Prefer the requested model, then installed preferred models, then the first installed model.
    raise NotImplementedError("Participant exercise: complete choose_model.")


OLLAMA_URL, OLLAMA_TAGS = choose_ollama_endpoint()
MODEL = choose_model(OLLAMA_TAGS)
SYSTEM_PROMPT = os.environ.get("G1_CHAT_SYSTEM", DEFAULT_SYSTEM_PROMPT)
TEMPERATURE = float(os.environ.get("G1_CHAT_TEMPERATURE", "0.4"))
TIMEOUT_S = float(os.environ.get("G1_CHAT_TIMEOUT", "30"))
MAX_HISTORY = int(os.environ.get("G1_CHAT_MAX_HISTORY", "4"))
NUM_PREDICT = int(os.environ.get("G1_CHAT_NUM_PREDICT", "48"))
NUM_CTX = int(os.environ.get("G1_CHAT_NUM_CTX", "1024"))
KEEP_ALIVE = os.environ.get("G1_CHAT_KEEP_ALIVE", "15m")
NUM_THREAD = os.environ.get("G1_CHAT_NUM_THREAD")
MIC_TOPIC = os.environ.get("G1_CHAT_MIC_TOPIC", "/audio_msg,/audio_msg/filter")
MIC_MIN_CONFIDENCE = float(os.environ.get("G1_CHAT_MIC_MIN_CONFIDENCE", "0.0"))
POST_SPEAK_IGNORE_S = float(os.environ.get("G1_CHAT_POST_SPEAK_IGNORE_S", "1.5"))
ANSWER_FILLERS = os.environ.get("G1_CHAT_ANSWER_FILLERS", "0") == "1"

messages = [{"role": "system", "content": SYSTEM_PROMPT}]
robot = None
speak_replies = False
mic_bridge = None
mic_executor = None
last_audio_index = None
last_audio_text = None
last_reply_text = None
last_reply_ts = 0.0


def post_ollama_chat(body, timeout=TIMEOUT_S):
    # TODO: POST JSON to Ollama /api/chat and handle HTTP errors with useful messages.
    raise NotImplementedError("Participant exercise: complete post_ollama_chat.")


def ask_ollama(user_text):
    # TODO: Append the user message, call the chat API, clean the reply, and maintain bounded history.
    raise NotImplementedError("Participant exercise: complete ask_ollama.")


def warm_up_ollama():
    # TODO: Send a tiny non-streaming prompt so the selected model is loaded before the demo.
    raise NotImplementedError("Participant exercise: complete warm_up_ollama.")

print(f"Ollama chat ready: url={OLLAMA_URL} model={MODEL} available={[m.get('name') for m in OLLAMA_TAGS.get('models', [])]}")


Optional robot speech binding. Set `enable_robot_speech = True` before running this cell if replies should be spoken.


In [ ]:
enable_robot_speech = False

if enable_robot_speech:
    robot = Robot(iface=IFACE, domain_id=DOMAIN_ID, safety_boot=False, auto_start_sensors=False)
    speak_replies = True
    print("Robot speech enabled.")
else:
    print("Robot speech disabled. Set enable_robot_speech=True and rerun this cell to speak replies.")


Run the chat panel. Typed prompts and microphone ASR prompts both use the same Ollama request path. The microphone subscriber listens to `/audio_msg` by default.


In [ ]:
prompt = widgets.Textarea(placeholder="Type a message...", layout=widgets.Layout(width="100%", height="90px"))
send = widgets.Button(description="Send", button_style="success")
warmup = widgets.Button(description="Warm Up")
clear = widgets.Button(description="Clear")
speak = widgets.Checkbox(value=speak_replies, description="speak replies")
mic_topic = widgets.Text(value=MIC_TOPIC, description="Mic topic", layout=widgets.Layout(width="320px"))
min_conf = widgets.FloatSlider(value=MIC_MIN_CONFIDENCE, min=0.0, max=1.0, step=0.05, description="Min conf")
answer_fillers = widgets.Checkbox(value=ANSWER_FILLERS, description="answer fillers")
start_mic = widgets.Button(description="Start Mic", button_style="info")
stop_mic = widgets.Button(description="Stop Mic")
chat_log = widgets.Textarea(layout=widgets.Layout(width="100%", height="420px"), disabled=True)


def add(line):
    # TODO: Complete the implementation for add using the surrounding notebook context.
    raise NotImplementedError("Participant exercise: complete add.")


def speak_reply(reply):
    # TODO: Optionally create the robot speech client and send the reply to robot.say().
    raise NotImplementedError("Participant exercise: complete speak_reply.")


def handle_user_text(text, source="you"):
    # TODO: Log user text, call the model, log the reply, and optionally speak it.
    raise NotImplementedError("Participant exercise: complete handle_user_text.")


def should_answer_audio(text, confidence, index):
    # TODO: Filter microphone transcripts by confidence, duplicates, fillers, and echo from recent replies.
    raise NotImplementedError("Participant exercise: complete should_answer_audio.")


class NotebookMicBridge(Node):
    def __init__(self, topics):
        # TODO: Initialize instance fields, clients, publishers/subscribers, locks, and default state needed by the class.
        raise NotImplementedError("Participant exercise: complete __init__.")

    def on_audio_msg(self, msg):
        # TODO: Parse each microphone message and forward accepted transcripts to the chat handler.
        raise NotImplementedError("Participant exercise: complete on_audio_msg.")


def on_send(_):
    # TODO: Read the prompt widget, clear it, and submit non-empty text to the chat handler.
    raise NotImplementedError("Participant exercise: complete on_send.")


def on_warmup(_):
    # TODO: Run the model warmup and append timing/status to the chat log.
    raise NotImplementedError("Participant exercise: complete on_warmup.")


def on_clear(_):
    # TODO: Clear visible chat history and reset conversation state as needed.
    raise NotImplementedError("Participant exercise: complete on_clear.")


def on_start_mic(_):
    # TODO: Initialize ROS, subscribe to the configured audio topics, and spin in a background thread.
    raise NotImplementedError("Participant exercise: complete on_start_mic.")


def on_stop_mic(_):
    # TODO: Stop the ROS executor, destroy the node, and clean up microphone state.
    raise NotImplementedError("Participant exercise: complete on_stop_mic.")

send.on_click(on_send)
warmup.on_click(on_warmup)
clear.on_click(on_clear)
start_mic.on_click(on_start_mic)
stop_mic.on_click(on_stop_mic)
display(widgets.VBox([
    prompt,
    widgets.HBox([send, warmup, clear, speak]),
    widgets.HBox([mic_topic, min_conf, answer_fillers, start_mic, stop_mic]),
    chat_log,
]))
